In [ ]:
import re
import ast
import numpy as np
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer, util
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv(
    "arxiv_data.csv",
    engine="python",
    quotechar='"',
    escapechar='\\',
    on_bad_lines='skip'
)

# rename to match pipeline
df["abstract"] = df["summaries"]

# terms is list-like string → take first category
import ast
import pandas as pd

def safe_first_category(x):
    if pd.isna(x):
        return "unknown"

    try:
        parsed = ast.literal_eval(str(x))

        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed[0]

        return "unknown"

    except:
        return "unknown"

df["category"] = df["terms"].apply(safe_first_category)

# =========================
# Hierarchical labels
# =========================
df["level1"] = df["category"].apply(lambda x: x.split(".")[0])
df["level2"] = df["category"]

freq = Counter(df["level2"])
MIN_SAMPLES = 20

def map_rare(cat):
    domain = cat.split(".")[0]
    return cat if freq[cat] >= MIN_SAMPLES else f"OTHER_{domain}"

df["level2_grouped"] = df["level2"].apply(map_rare)

# SBERT embeddings
# =========================
sbert = SentenceTransformer("all-mpnet-base-v2")
X = sbert.encode(df["abstract"].tolist(), show_progress_bar=True)

# =========================
# Level 1 classifier
# =========================
X_train, X_test, y1_train, y1_test = train_test_split(
    X, df["level1"], test_size=0.2, random_state=42
)

lvl1 = LogisticRegression(max_iter=3000, class_weight="balanced")
lvl1.fit(X_train, y1_train)

# =========================
# Level 2 classifiers
# =========================
lvl2_models = {}
trusted_refs = {}

for domain in df["level1"].unique():
    sub_df = df[df["level1"] == domain]
    if sub_df["level2_grouped"].nunique() < 2:
        continue

    X_sub = sbert.encode(sub_df["abstract"].tolist())
    y_sub = sub_df["level2_grouped"]

    clf = LogisticRegression(max_iter=3000, class_weight="balanced")
    clf.fit(X_sub, y_sub)

    lvl2_models[domain] = clf
    trusted_refs[domain] = X_sub[:50]

# =========================
# Writing quality
# =========================
def writing_quality(text):
    words = text.split()
    sentence_count = max(1, len(re.split(r"[.!?]", text)) - 1)

    avg_sent_len = len(words) / sentence_count
    citation_count = len(re.findall(r"\[\d+\]|\(\w+, \d{4}\)", text))

    score = 0.5

    if 8 <= avg_sent_len <= 35:
        score += 0.2
    if len(words) > 80:
        score += 0.15
    if citation_count > 0:
        score += 0.15

    return min(score, 1.0)

# =========================
# Advanced credibility score
# =========================
def advanced_credibility_score(text):
    emb = sbert.encode([text])

    pred_lvl1 = lvl1.predict(emb)[0]
    lvl1_prob = lvl1.predict_proba(emb).max()

    score = 0.35 * lvl1_prob

    if pred_lvl1 in lvl2_models:
        clf2 = lvl2_models[pred_lvl1]
        pred_lvl2 = clf2.predict(emb)[0]
        lvl2_prob = clf2.predict_proba(emb).max()

        score += 0.25 * lvl2_prob

        if "OTHER" in pred_lvl2:
            score -= 0.15

    score += 0.2 * writing_quality(text)

    if pred_lvl1 in trusted_refs:
        sim = util.cos_sim(emb, trusted_refs[pred_lvl1]).max().item()
        score += 0.2 * sim

    return round(float(np.clip(score, 0, 1)), 3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1639 [00:00<?, ?it/s]